In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

print("Loading dataset from Hugging Face...")
ds = load_dataset("Exqrch/IndoDiscourse", "main")

df = ds['main'].to_pandas() 

print(f"Total data awal: {len(df)} baris")

def check_if_toxic(labels):
    if isinstance(labels, (list, np.ndarray)):
        return '1' in labels or 1 in labels
    return False

df['is_toxic'] = df['toxicity'].apply(check_if_toxic)
df_toxic = df[df['is_toxic'] == True].copy()

print(f"Total data dengan minimal satu label toxic: {len(df_toxic)} baris\n")

# Distribusi Topik pada Teks Toxic
plt.figure(figsize=(12, 6))
topic_counts = df_toxic['topic'].value_counts()
sns.barplot(x=topic_counts.values, y=topic_counts.index, palette='Reds_r')
plt.title('Topik yang Paling Banyak Memiliki Komentar/Teks Toxic')
plt.xlabel('Jumlah Teks Toxic')
plt.ylabel('Topik')
plt.tight_layout()
plt.show()

# Korelasi dengan Label Negatif Lain
other_negative_labels = [
    'profanity_obscenity', 
    'threat_incitement_to_violence', 
    'insults', 
    'identity_attack', 
    'sexually_explicit'
]

co_occurrence = {}
for label in other_negative_labels:
    count_has_label = df_toxic[label].apply(check_if_toxic).sum()
    co_occurrence[label] = count_has_label

co_df = pd.DataFrame(list(co_occurrence.items()), columns=['Tipe Pelanggaran', 'Jumlah'])
co_df = co_df.sort_values(by='Jumlah', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=co_df, x='Jumlah', y='Tipe Pelanggaran', palette='magma')
plt.title('Irisan Teks Toxic dengan Kategori Negatif Lainnya')
plt.xlabel('Jumlah Teks')
plt.tight_layout()
plt.show()

# Distribusi Panjang Karakter pada Teks Toxic
df_toxic['text_length'] = df_toxic['text'].fillna('').apply(len)

plt.figure(figsize=(10, 5))
sns.histplot(df_toxic['text_length'], bins=50, kde=True, color='red')
plt.title('Distribusi Panjang Karakter pada Teks Toxic')
plt.xlabel('Panjang Karakter')
plt.ylabel('Frekuensi')
plt.show()

# Top 15 Kata dan Frasa dalam Teks Toxic
def get_top_ngrams(corpus, n=1, top_k=15):
    vec = CountVectorizer(ngram_range=(n, n)).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:top_k]

corpus_toxic = df_toxic['text'].dropna().tolist()

print("\n=== TOP 15 KATA (UNIGRAM) DALAM TEKS TOXIC ===")
top_1_words = get_top_ngrams(corpus_toxic, n=1, top_k=15)
for word, freq in top_1_words:
    print(f"- {word}: {freq}")

print("\n=== TOP 15 FRASA (BIGRAM) DALAM TEKS TOXIC ===")
top_2_words = get_top_ngrams(corpus_toxic, n=2, top_k=15)
for phrase, freq in top_2_words:
    print(f"- {phrase}: {freq}")

[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]
